# CVaR optimal vs equal weighted portfolios.

The following optimization are considered:
1. optimal portfolio with maximum C-Sharpe ratio - 'P_Sharpe'
2. optimal portfolio with minimum CVaR - 'P_MinRisk'
3. optimal CVaR portfolio with expected rate of returns (quarterly) of 0.032 - 'P_Risk'
4. optimal CVaR portfolio with risk aversion coefficient of 0.5 - 'P_RiskAverse'
5. optimal CVaR portfolio with same risk as the equal weighted portfolio - 'P_InvNrisk'
6. same as 1 but using the minimization of inverse C-Sharpe ratio - 'P_Sharpe2'

The equal weighted portfolio is named 'P_N'.

The rebalancing schedule is identical for all portfolios.

We start by importing **azapy** and other useful packages.

In [1]:
import azapy as az

print(f"azapy version {az.version()} >= 1.2.0", flush=True)

azapy version 1.2.6 >= 1.2.0


### Collect historical market data

Note the flag `force=False`. The function will attempt first to read the market data from the local directory `mktdir`. If that fails, then it will access the data provider servers *(in this case yahoo)*.

>Make sure that `mktdir` holds a convenient location to save the market data.
>You can inhibit the saving mechanism by setting `save=False` in the call of `az.readMkT` function (_see `readMkT` documentation_ https://azapy.readthedocs.io/en/latest/).

In [2]:
symb = ['GLD', 'TLT', 'XLV', 'SPY', 'VHT']

sdate = "2012-01-01"
edate = 'today'
mktdir = "../MkTdata"

mktdata = az.readMkT(symb, sdate=sdate, edate=edate, file_dir=mktdir)

read GLD data from file
read TLT data from file
read XLV data from file
read SPY data from file
read VHT data from file

Request between 2012-01-03 : 2026-05-06
                    GLD         TLT         XLV         SPY         VHT
source            yahoo       yahoo       yahoo       yahoo       yahoo
force             False       False       False       False       False
save               True        True        True        True        True
file_dir     ../MkTdata  ../MkTdata  ../MkTdata  ../MkTdata  ../MkTdata
file_format         csv         csv         csv         csv         csv
api_key            None        None        None        None        None
nrow               3606        3606        3606        3606        3606
sdate        2012-01-03  2012-01-03  2012-01-03  2012-01-03  2012-01-03
edate        2026-05-06  2026-05-06  2026-05-06  2026-05-06  2026-05-06
error                No          No          No          No          No
extraction time 0.256 s


### Set dispersion measure parameters

Set the parameters for a mCVaR (mixture of 3 CVaR's) dispersion:
- `alpha` the list of CVaR confidence levels,
- `coef` the list of mixture coefficients.

In [3]:
alpha = [0.95, 0.90, 0.85]
coef = [0.1, 0.3, 0.6]
hlength = 3.25
verbose = False

### Define a dictionary of portfolio parameters

The index is the name of the portfolio, and the values are dictionaries of model parameters:
 - 'type' is the portfolio class name, 
 - 'm_param' is a dictionary of parameters required by the corresponding `set_model` function.
 
 We had adopted this rather encrypted method to facilitate an easy call to the model classes.

In [4]:
models = {'P_Sharpe': {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'Sharpe', 'mu0': 0, 'hlength': hlength, 'verbose': verbose}},
          'P_Risk': {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'Risk', 'mu': 0.035, 'hlength': hlength, 'verbose': verbose}},
          'P_MinRisk': {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'MinRisk', 'hlength': hlength, 'verbose': verbose}},
          'P_InvNrisk': {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'InvNrisk', 'hlength': hlength, 'verbose': verbose}},
          'P_RiskAverse': {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'RiskAverse', 'aversion':0.5, 'hlength': hlength, 'verbose': verbose}},
          'P_Diverse': {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'Diverse', 'mu': 0.035, 'hlength': hlength, 'verbose': verbose}},
          'P_MaxDiverse' : {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'MaxDiverse', 'hlength': hlength, 'verbose': verbose}},
          'P_InvNdiverse' : {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'InvNdiverse', 'hlength': hlength, 'verbose': verbose}},
          'P_InvNdrr' : {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'InvNdrr', 'hlength': hlength, 'verbose': verbose}},
          'P_N': {'type': 'Port_ConstW', 'm_param': {'ww': None}}}

### Main computation loop

The portfolios time-series are stored in a list while the actual model classes are stored in a dictionary with the index given by the portfolio name. They can be interrogated later for individual computed results.

In [5]:
port = []
pp = {}
for key, val in models.items():
    ppz = getattr(az, val['type'])
    pp_ = ppz(mktdata, pname=key)
    pp[key] = pp_
    port_ = pp_.set_model(**val['m_param'])
    port.append(port_)

### Build a comparison environment 

Using the list of individual portfolio time-series, `port`, we can set a `Port_Simple` class to facilitate the visual and numerical comparisons. We are interested in comparing the components of this portfolio of portfolios and ignore their aggregated time-series. 

>Note the call to `set_model` method that is a must.

>Observation: `Port_Simple` is the class that supports the back testing of "Buy and Hold" portfolio (_see its documentation_).
It also can be used as a tool to compare the performance of multiple portfolios. Here we use it in this latter capacity.

In [6]:
ps = az.Port_Simple(port, col='close', pname='ALL')
_ = ps.set_model()

### Visualize the portfolio time-series

We had used the following flags:
- `componly=True` to plot only the initial portfolio time-series without their aggregated portfolio,
- `fancy=True` to use the interactive `plotly` time-series library.

In [7]:
_ = ps.port_view_all(sdate='2000-01-01', componly=True, fancy=True, title="Relative performance")

### Portfolio performance comparisons 

- `RR` is the average annual portfolio rate of returns. 
- `DD` is the maximum drawdown rate.
- `Beta` is the ratio `RR/DD`.
- `DD_date` is the date of maximum drawdown.
- `DD_start` is the date when the maximum drawdown event had started.
- `DD_end` is the date when the maximum drawdown event had ended. If it is `nan` then the drawdown is still in progress (`DD` and `DD_date` are only provisional).

We had used the same flags as before.

In [8]:
ps.port_perf(componly=True, fancy=True)

,RR,DD,RoMaD,DD_date,DD_start,DD_end,DD_days
symbol,,,,,,,
P_N,9.04,-19.16,0.471885,2022-10-20,2021-12-30,2024-02-23,785
P_Sharpe,9.67,-22.50,0.429687,2022-09-27,2021-11-12,2024-05-21,921
P_InvNrisk,9.17,-22.86,0.401207,2022-09-27,2021-11-17,2024-06-20,946
P_MinRisk,7.87,-20.64,0.381420,2022-10-20,2021-12-31,2024-03-27,817
P_RiskAverse,8.39,-22.30,0.376252,2022-09-27,2021-11-12,2024-03-27,866
P_InvNdiverse,8.58,-24.50,0.350364,2022-10-20,2021-11-17,2024-05-17,912
P_Risk,11.13,-33.82,0.329130,2020-03-23,2020-02-19,2020-06-08,110
P_Diverse,10.21,-33.89,0.301224,2020-03-23,2020-02-19,2021-03-17,392
P_InvNdrr,6.78,-25.35,0.267326,2022-10-20,2021-11-18,2024-08-19,1005


### Portfolio annual returns

We had used the same flags as before.

In [9]:
ps.port_annual_returns(withcomp=True, componly=True, fancy=True)

symbol,P_Diverse,P_InvNdiverse,P_InvNdrr,P_InvNrisk,P_MaxDiverse,P_MinRisk,P_N,P_Risk,P_RiskAverse,P_Sharpe
year,,,,,,,,,,
2015,-3.36%,-3.80%,-4.39%,-4.76%,-4.67%,-4.04%,-3.48%,-2.27%,-5.97%,-4.67%
2016,-1.21%,0.06%,-0.34%,-0.19%,2.44%,5.19%,3.88%,0.20%,-1.44%,0.43%
2017,17.30%,17.26%,16.71%,18.50%,16.87%,17.03%,17.81%,17.30%,17.30%,17.39%
2018,0.56%,0.62%,4.60%,-0.25%,0.37%,0.02%,3.79%,0.62%,-0.70%,-0.15%
2019,26.53%,23.71%,22.70%,23.91%,23.20%,23.08%,18.92%,26.34%,23.68%,23.74%
2020,-0.91%,10.01%,12.12%,9.73%,18.32%,14.80%,21.47%,-0.55%,11.72%,11.62%
2021,19.76%,11.51%,6.44%,9.86%,6.33%,5.35%,12.11%,20.26%,5.09%,5.13%
2022,-16.20%,-16.03%,-17.84%,-11.99%,-19.66%,-12.39%,-11.95%,-11.15%,-12.05%,-12.36%
2023,7.91%,9.02%,8.20%,5.89%,7.38%,7.13%,10.14%,7.94%,6.54%,4.73%


### Portfolios monthly returns

We had used the following flags:
- `withcomp = True` to print the portfolio returns,
- `componly = True` to exclude the aggregated portfolio of portfolios,
- `fancy = True` to print the rates in a percent format.

In [10]:
ps.port_monthly_returns(withcomp=True, componly=True, fancy=True)

### Example of a specific portfolio performance inquiry

In [11]:
pp['P_Sharpe'].port_monthly_returns(fancy=True)

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
month,,,,,,,,,,,,
1,nan%,-3.28%,1.44%,2.76%,6.41%,2.05%,-1.50%,-4.23%,-0.25%,2.63%,6.65%,7.07%
2,nan%,0.62%,3.01%,-3.52%,2.08%,-3.57%,-4.59%,-1.13%,-4.80%,3.62%,1.61%,4.35%
3,nan%,3.67%,0.28%,0.96%,-2.02%,-0.69%,-0.06%,-0.31%,3.12%,2.51%,5.63%,-11.54%
4,nan%,1.08%,1.11%,-0.25%,2.42%,7.43%,3.78%,-5.71%,2.58%,-3.91%,4.39%,4.03%
5,nan%,1.52%,1.50%,2.26%,-3.62%,1.84%,3.62%,-1.79%,-3.56%,2.55%,0.32%,1.91%
6,-1.91%,3.67%,-0.44%,-1.03%,6.39%,-1.71%,-1.73%,-3.31%,3.11%,1.55%,1.33%,nan%
7,3.11%,3.43%,1.10%,2.13%,1.04%,7.02%,3.21%,0.29%,1.50%,3.76%,-0.31%,nan%
8,-7.16%,-1.77%,1.22%,2.60%,1.33%,-0.98%,1.08%,-4.38%,-0.89%,3.83%,4.67%,nan%
9,-7.55%,-0.05%,1.74%,-0.41%,0.40%,-1.63%,-4.05%,-3.38%,-4.23%,0.68%,9.87%,nan%


In [12]:
pp['P_Sharpe'].port_annual_returns(withcomp=True, fancy=True)

,P_Sharpe,GLD,SPY,TLT,VHT,XLV
year,,,,,,
2015,-4.67%,-9.77%,-1.77%,5.03%,-5.74%,-4.60%
2016,0.43%,8.03%,12.00%,1.17%,-3.21%,-2.76%
2017,17.39%,12.81%,21.71%,9.18%,23.26%,21.77%
2018,-0.15%,-1.94%,-4.57%,-1.61%,5.58%,6.28%
2019,23.74%,17.86%,31.22%,14.12%,21.87%,20.45%
2020,11.62%,24.81%,18.33%,18.15%,18.29%,13.30%
2021,5.13%,-4.15%,28.73%,-4.60%,20.57%,26.04%
2022,-12.36%,-0.77%,-18.18%,-31.23%,-5.60%,-2.08%
2023,4.73%,12.69%,26.18%,2.77%,2.52%,2.07%
